# 05.14 - Imbalanced Learning

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Imbalanced data has one class far more common than another (e.g., 95% non-fraud, 5% fraud). Models trained on such data tend to ignore the minority class. We learn to handle this with resampling, class weights, and appropriate metrics.

## 2. Why Does This Matter?

Fraud detection, disease screening, and rare-event prediction all face imbalance. Handling it correctly is critical for real-world impact.

## 3. Prerequisites

- Unit 05.4 (Model Evaluation)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Identify the problem with imbalanced data
- Use resampling (oversampling/undersampling)
- Use class weights
- Choose appropriate metrics

## 5. Mental Model

The problem: accuracy is misleading. A model predicting the majority class gets high accuracy but misses the minority.

Solutions:
1. **Resampling**: oversample minority or undersample majority.
2. **Class weights**: penalize errors on minority more.
3. **Metrics**: use precision/recall/F1/ROC-AUC, not accuracy.


## 6. Generate Imbalanced Data

Create a dataset with 95% majority class.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=10, weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Class balance in train: {y_train.mean():.3f}")
print(f"Minority class samples: {(y_train == 1).sum()}")


Class balance in train: 0.053
Minority class samples: 74


## 7. Baseline: Ignoring Imbalance

Train a plain logistic regression.


In [2]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.3f}")
print("\nAccuracy is high but recall is low - the model misses positives.")


Accuracy:  0.958
Precision: 0.652
Recall:    0.469
F1:        0.545
ROC-AUC:   0.928

Accuracy is high but recall is low - the model misses positives.


## 8. Solution 1: Class Weights

Penalize errors on the minority class more.


In [3]:
model_w = LogisticRegression(max_iter=1000, class_weight='balanced')
model_w.fit(X_train, y_train)
y_pred_w = model_w.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_w):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_w):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_w):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_w):.3f}")
print("\nClass weights improve recall significantly.")


Accuracy:  0.883
Precision: 0.293
Recall:    0.844
F1:        0.435

Class weights improve recall significantly.


## 9. Solution 2: Oversampling (SMOTE)

Synthesize minority samples.


In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print(f"After SMOTE: {X_res.shape[0]} samples, balance={y_res.mean():.3f}")

model_sm = LogisticRegression(max_iter=1000).fit(X_res, y_res)
y_pred_sm = model_sm.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_sm):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_sm):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_sm):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_sm):.3f}")
print("\nSMOTE balances the classes before training.")


After SMOTE: 2652 samples, balance=0.500
Accuracy:  0.887
Precision: 0.291
Recall:    0.781
F1:        0.424

SMOTE balances the classes before training.


## 10. Solution 3: Undersampling

Remove majority samples to balance.


In [5]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_und, y_und = rus.fit_resample(X_train, y_train)
print(f"After undersampling: {X_und.shape[0]} samples, balance={y_und.mean():.3f}")

model_und = LogisticRegression(max_iter=1000).fit(X_und, y_und)
y_pred_und = model_und.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_und):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_und):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_und):.3f}")
print(f"F1:        {f1_score(y_test, y_pred_und):.3f}")
print("\nUndersampling discards majority data.")


After undersampling: 148 samples, balance=0.500
Accuracy:  0.858
Precision: 0.252
Recall:    0.844
F1:        0.388

Undersampling discards majority data.


## 11. Compare All Approaches

Summary of F1 and ROC-AUC across methods.


In [6]:
print(f"{'Method':<20} {'F1':<8} {'ROC-AUC':<8}")
print("-" * 40)
methods = [
    ("Baseline", f1_score(y_test, y_pred), roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])),
    ("Class weights", f1_score(y_test, y_pred_w), roc_auc_score(y_test, model_w.predict_proba(X_test)[:, 1])),
    ("SMOTE", f1_score(y_test, y_pred_sm), roc_auc_score(y_test, model_sm.predict_proba(X_test)[:, 1])),
    ("Undersample", f1_score(y_test, y_pred_und), roc_auc_score(y_test, model_und.predict_proba(X_test)[:, 1])),
]
for name, f1, auc in methods:
    print(f"{name:<20} {f1:<8.3f} {auc:<8.3f}")
print("\nAll methods improve over the baseline on F1/ROC-AUC.")


Method               F1       ROC-AUC 
----------------------------------------
Baseline             0.545    0.928   
Class weights        0.435    0.928   
SMOTE                0.424    0.930   
Undersample          0.388    0.926   

All methods improve over the baseline on F1/ROC-AUC.


## 12. Failure Case: Accuracy on Imbalanced Data

Accuracy alone hides the problem.


In [7]:
print("A model predicting all 0s would get:")
print(f"  Accuracy = {1 - y_test.mean():.3f} (looks great!)")
print(f"  Recall   = 0.000 (misses all positives)")
print("\nAlways use precision/recall/F1/ROC-AUC for imbalanced data.")


A model predicting all 0s would get:
  Accuracy = 0.947 (looks great!)
  Recall   = 0.000 (misses all positives)

Always use precision/recall/F1/ROC-AUC for imbalanced data.


## 13. Debugging: Common Errors

- **Using accuracy**: misleading.
- **Not stratifying splits**: imbalance varies across folds.
- **SMOTE on test set**: leakage.

## 14. Real-World Considerations

- Resample only the training set, never test.
- Use stratified CV.
- Choose the metric based on the cost of errors.

## 15. Common Mistakes

- SMOTE before splitting (leakage).
- Reporting only accuracy.

## 16. When NOT to Use

- When classes are roughly balanced.
- When you need calibrated probabilities (resampling distorts them).

## 17. Challenge

Use a tree-based model with class weights and compare to logistic regression.


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Random forest (class_weight): F1={f1_score(y_test, y_pred_rf):.3f}, "
      f"ROC-AUC={roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]):.3f}")
print(f"Logistic (class_weight):      F1={f1_score(y_test, y_pred_w):.3f}, "
      f"ROC-AUC={roc_auc_score(y_test, model_w.predict_proba(X_test)[:, 1]):.3f}")
print("\nTree models also benefit from class weights.")


Random forest (class_weight): F1=0.580, ROC-AUC=0.902
Logistic (class_weight):      F1=0.435, ROC-AUC=0.928

Tree models also benefit from class weights.


## 18. Closed-Book Recall

Without looking back:

1. Why is accuracy misleading on imbalanced data?
2. What is SMOTE?
3. What do class weights do?
4. Why resample only the training set?

## 19. Teach-Back Questions

Explain to another person:

- The problem with imbalanced data.
- The tradeoffs of oversampling vs class weights.

## 20. Summary

You handled imbalanced data with class weights, SMOTE, and undersampling, and learned to use appropriate metrics.

## 21. Further Experiment

- Try SMOTE variants (BorderlineSMOTE).
- Tune the decision threshold.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, imbalanced-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
